# Detector Anomaly

система обнаружения аномалий для шести контролируемых каналов X06–X11.



In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from IPython.display import display

ROOT = Path.cwd()
PUBLIC_COLUMNS = ['timestamp', *[f'X{index:02d}' for index in range(1, 13)]]
TARGETS = ['X06', 'X07', 'X08', 'X09', 'X10', 'X11']
BRANCHES = ('strict_robust', 'full_telemetry')
targets = list(TARGETS)
branches = list(BRANCHES)

DATA_ROOT_CANDIDATES = (
    ROOT / 'data/v2',
    ROOT.parent / 'Archive/pre_minimal_refactor/internal_legacy/data/v2',
)
DATA_ROOT = next(
    (path for path in DATA_ROOT_CANDIDATES if (path / 'development_train').is_dir()),
    DATA_ROOT_CANDIDATES[0],
)
METRICS_ROOT_CANDIDATES = (
    ROOT / 'artifacts/work/metrics',
    ROOT.parent / 'Archive/pre_minimal_refactor/internal_legacy/artifacts/work/metrics',
)
METRICS_ROOT = next((path for path in METRICS_ROOT_CANDIDATES if path.is_dir()), None)
SPLIT_DIRS = {
    'train': DATA_ROOT / 'development_train',
    'validation': DATA_ROOT / 'development_validation',
    'final_test': DATA_ROOT / 'final_unseen_test',
}
ROOT_MANIFEST_PATH = DATA_ROOT / 'manifests/data_v2_manifest.json'
ARTIFACT_NAMES = ('observed.parquet', 'masks.parquet', 'events.parquet')
RUN_TRAINING = False
EXPORT_MODELS = False


In [2]:
QUANTILE_SETS = {
    'q90': (0.05, 0.50, 0.95),
    'q98': (0.01, 0.50, 0.99),
}
TRAIN_SAMPLING_SECONDS = 5
CALIBRATION_FRACTION = 0.50
MINIMUM_GROUP_ROWS = 5_000
LINEAR_PARAMS = {
    'epochs': 10,
    'batch_size': 65_536,
    'learning_rate': 0.08,
    'l2': 1e-4,
}
CATBOOST_PARAMS = {
    'iterations': 60,
    'depth': 5,
    'learning_rate': 0.08,
    'l2_leaf_reg': 5.0,
    'random_strength': 1.0,
    'thread_count': 1,
    'random_seed': 20260913,
}
CALIBRATION_METHODS = (
    'global_symmetric_conformal',
    'operating_mode_mondrian',
)



## 1. Данные

Полный набор разбит на обучающую, валидационную и итоговую тестовую выборки. Для каждой выборки используются связанные файлы `observed.parquet`, `masks.parquet` и `events.parquet`; `data/demo.csv` в обучении и оценке качества не участвует.


In [3]:
# проверяем доступность набора до обучения
development_paths = {
    split: {
        name: folder / name
        for name in ARTIFACT_NAMES
    }
    for split, folder in SPLIT_DIRS.items()
}

development_available = all(
    path.is_file()
    for split in ('train', 'validation')
    for path in development_paths[split].values()
)
data_available = development_available and ROOT_MANIFEST_PATH.is_file()
training_enabled = RUN_TRAINING and data_available


## 2. Временное разделение обучающей, валидационной и итоговой тестовой выборок

Ряды обрабатываются в хронологическом порядке, без перемешивания. Обучающая выборка прореживается до шага 5 секунд только перед обучением. Первая половина валидационной выборки используется для калибровки, вторая — для выбора итоговой модели. Итоговая тестовая выборка используется только после завершения выбора.


In [4]:
def _load_split(split):
    folder = SPLIT_DIRS[split]
    observed = pd.read_parquet(folder / 'observed.parquet')
    masks = pd.read_parquet(folder / 'masks.parquet')
    events = pd.read_parquet(folder / 'events.parquet')
    observed['timestamp'] = pd.to_datetime(observed['timestamp'], utc=True)
    masks['timestamp'] = pd.to_datetime(masks['timestamp'], utc=True)
    assert len(observed) == len(masks)
    assert observed['timestamp'].equals(masks['timestamp'])

    source_channels = [column for column in observed.columns if column != 'timestamp']
    channel_names = dict(zip(source_channels, PUBLIC_COLUMNS[1:]))
    public_observed = observed.rename(columns=channel_names)
    public_masks = masks.copy()
    public_masks['sensor_fault_channel'] = public_masks['sensor_fault_channel'].map(channel_names)
    public_events = events.copy()
    for column in ('sensor', 'channel'):
        public_events[column] = public_events[column].map(channel_names)

    frame = public_observed.copy()
    for column in (
        'split',
        'label_state',
        'sensor_fault_active',
        'sensor_fault_channel',
        'process_event_active',
        'process_event_type',
    ):
        frame[column] = public_masks[column].to_numpy()
    return frame, public_masks, public_events

if training_enabled:
    train, train_masks, train_events = _load_split('train')
    validation, validation_masks, validation_events = _load_split('validation')
    final_test = final_test_masks = final_test_events = None

    train_clean = train[train['label_state'].eq('CLEAN')].copy()
    elapsed = (
        train_clean['timestamp'] - train_clean['timestamp'].iloc[0]
    ).dt.total_seconds().astype('int64')
    train_fit = train_clean.loc[
        elapsed % TRAIN_SAMPLING_SECONDS == 0
    ].reset_index(drop=True)
    assert train['timestamp'].max() < validation['timestamp'].min()
else:
    train = validation = final_test = None
    train_masks = validation_masks = final_test_masks = None
    train_events = validation_events = final_test_events = None
    train_fit = None


## 3. Целевые каналы и наборы признаков

Strict использует только X01, X02, X03. Full использует наблюдаемые X-каналы кроме собственного целевого канала. Скрытые состояния, маски неисправностей и события в признаки не попадают.


In [5]:
visible_features = PUBLIC_COLUMNS[1:]
feature_policies = {
    target: {
        'strict_robust': ['X01', 'X02', 'X03'],
        'full_telemetry': [feature for feature in visible_features if feature != target],
    }
    for target in targets
}
policy_table = pd.DataFrame([
    {
        'target': target,
        'branch': branch,
        'features': ', '.join(feature_policies[target][branch]),
        'n_features': len(feature_policies[target][branch]),
    }
    for target in targets
    for branch in branches
])
display(policy_table)
assert all(
    feature_policies[target]['strict_robust'] == ['X01', 'X02', 'X03']
    for target in targets
)
assert all(
    target not in feature_policies[target][branch]
    for target in targets
    for branch in branches
)
assert set().union(
    *(set(feature_policies[target][branch]) for target in targets for branch in branches)
) <= set(visible_features)


,target,branch,features,n_features
0,X06,strict_robust,"X01, X02, X03",3
1,X06,full_telemetry,"X01, X02, X03, X04, X05, X07, X08, X09, X10, X...",11
2,X07,strict_robust,"X01, X02, X03",3
3,X07,full_telemetry,"X01, X02, X03, X04, X05, X06, X08, X09, X10, X...",11
4,X08,strict_robust,"X01, X02, X03",3
5,X08,full_telemetry,"X01, X02, X03, X04, X05, X06, X07, X09, X10, X...",11
6,X09,strict_robust,"X01, X02, X03",3
7,X09,full_telemetry,"X01, X02, X03, X04, X05, X06, X07, X08, X10, X...",11
8,X10,strict_robust,"X01, X02, X03",3
9,X10,full_telemetry,"X01, X02, X03, X04, X05, X06, X07, X08, X09, X...",11


## 4. Кандидаты, квантили и калибровка

Линейная квантильная регрессия получает стандартизацию только по обучающей выборке и кодирование X01 отдельными индикаторами. CatBoost MultiQuantile строит нижнюю, центральную и верхнюю границы. Калибровка расширяет границы на отдельной части валидационной выборки.


In [ ]:
# собирает матрицу признаков для регрессии
def _linear_design(frame, features, state=None):
    numeric = [feature for feature in features if feature != "X01"]
    if state is None:
        means = {feature: float(frame[feature].mean()) for feature in numeric}
        stds = {}
        for feature in numeric:
            values = frame[feature].to_numpy(dtype=np.float64)
            spread = float(values.std())
            stds[feature] = spread if spread > 0 else 1.0
        categories = (
            sorted(int(value) for value in frame["X01"].unique())
            if "X01" in features
            else []
        )
        state = {
            "numeric": numeric,
            "means": means,
            "stds": stds,
            "categories": categories,
        }
    parts = [np.ones((len(frame), 1), dtype=np.float64)]
    if numeric:
        values = frame[numeric].to_numpy(dtype=np.float64)
        parts.append(np.column_stack([
            (values[:, index] - state["means"][feature]) / state["stds"][feature]
            for index, feature in enumerate(numeric)
        ]))
    if state["categories"]:
        modes = frame["X01"].to_numpy(dtype=np.int64)
        parts.append(np.column_stack([
            (modes == category).astype(np.float64)
            for category in state["categories"][1:]
        ]))
    return np.column_stack(parts), state


def fit_linear(train_frame, target, features, levels):
    design, state = _linear_design(train_frame, features)
    actual = train_frame[target].to_numpy(dtype=np.float64)
    ols, _, _, _ = np.linalg.lstsq(design, actual, rcond=None)
    coefficients = np.repeat(ols[:, None], 3, axis=1)
    residual = actual[:, None] - design @ coefficients
    for index, level in enumerate(levels):
        coefficients[0, index] += np.quantile(residual[:, index], level)
    quantiles = np.asarray(levels, dtype=np.float64)
    for epoch in range(LINEAR_PARAMS["epochs"]):
        step = LINEAR_PARAMS["learning_rate"] / np.sqrt(epoch + 1.0)
        for start in range(0, len(actual), LINEAR_PARAMS["batch_size"]):
            stop = min(start + LINEAR_PARAMS["batch_size"], len(actual))
            batch_design = design[start:stop]
            batch_actual = actual[start:stop]
            residual = batch_actual[:, None] - batch_design @ coefficients
            gradient = -(
                batch_design.T
                @ (quantiles[None, :] - (residual < 0).astype(np.float64))
            ) / len(batch_actual)
            gradient[1:] += LINEAR_PARAMS["l2"] * coefficients[1:]
            coefficients -= step * gradient
    return {
        "kind": "linear",
        "state": state,
        "coefficients": coefficients,
    }


def fit_catboost(train_frame, target, features, levels):
    params = dict(CATBOOST_PARAMS)
    params.update({
        "loss_function": "MultiQuantile:alpha=" + ",".join(f"{level:g}" for level in levels),
        "verbose": False,
        "allow_writing_files": False,
    })
    model = CatBoostRegressor(**params)
    categorical = [index for index, feature in enumerate(features) if feature == "X01"]
    model.fit(train_frame[features], train_frame[target], cat_features=categorical)
    return {"kind": "catboost", "model": model}


def fit_candidate(train_frame, target, features, model_type, levels):
    if model_type == "linear_quantile":
        return fit_linear(train_frame, target, features, levels)
    return fit_catboost(train_frame, target, features, levels)


# возвращает границы и маску валидных строк
def predict_candidate(fitted, frame_part, features):
    valid = np.isfinite(frame_part[features].to_numpy(dtype=np.float64)).all(axis=1)
    result = np.full((len(frame_part), 3), np.nan, dtype=np.float64)
    if valid.any():
        if fitted["kind"] == "linear":
            design, _ = _linear_design(frame_part.loc[valid], features, fitted["state"])
            raw = design @ fitted["coefficients"]
        else:
            raw = np.asarray(
                fitted["model"].predict(frame_part.loc[valid, features]),
                dtype=np.float64,
            )
            if raw.ndim == 1:
                raw = raw.reshape(-1, 3)
        result[valid] = np.sort(raw, axis=1)
    return result, valid


In [ ]:
# считает отклонение от границ
def _calibration_scores(actual, predictions):
    return np.maximum.reduce([
        predictions[:, 0] - actual,
        actual - predictions[:, 2],
        np.zeros(len(actual)),
    ])


# подбирает Global или Mondrian (калибровка глобальлная или в зависимости от режима)
def fit_calibration(actual, predictions, modes, method, alpha):
    scores = _calibration_scores(actual, predictions)
    level = min(1.0, (1.0 - alpha) * (len(scores) + 1.0) / len(scores))
    global_expansion = float(np.quantile(scores, level, method="higher"))
    result = {
        "method": method,
        "alpha": alpha,
        "global_expansion": global_expansion,
        "group_expansions": {},
        "fallback_groups": [],
    }
    if method == "operating_mode_mondrian":
        for mode in sorted(np.unique(modes)):
            group = modes == mode
            if int(group.sum()) < MINIMUM_GROUP_ROWS:
                result["fallback_groups"].append(int(mode))
                continue
            group_scores = scores[group]
            group_level = min(
                1.0,
                (1.0 - alpha) * (len(group_scores) + 1.0) / len(group_scores),
            )
            result["group_expansions"][str(int(mode))] = float(
                np.quantile(group_scores, group_level, method="higher")
            )
    return result


# применяет calibration без изменения модели
def apply_calibration(predictions, modes, calibration):
    result = predictions.copy()
    expansions = np.full(
        len(result),
        calibration["global_expansion"],
        dtype=np.float64,
    )
    for mode, expansion in calibration["group_expansions"].items():
        expansions[modes == int(mode)] = expansion
    result[:, 0] -= expansions
    result[:, 2] += expansions
    return result


# считает coverage, width, interval score
def interval_metrics(actual, predictions, alpha):
    valid = np.isfinite(actual) & np.isfinite(predictions).all(axis=1)
    actual = actual[valid]
    predictions = predictions[valid]
    lower, center, upper = predictions.T
    width = upper - lower
    score = width + (2.0 / alpha) * np.maximum(lower - actual, 0.0)
    score += (2.0 / alpha) * np.maximum(actual - upper, 0.0)
    return {
        "coverage": float(np.mean((actual >= lower) & (actual <= upper))),
        "outside_rate": float(np.mean((actual < lower) | (actual > upper))),
        "mean_interval_width": float(np.mean(width)),
        "mean_interval_score": float(np.mean(score)),
        "center_mae": float(np.mean(np.abs(actual - center))),
        "center_rmse": float(np.sqrt(np.mean(np.square(actual - center)))),
    }



def select_candidate(group, coverage_tolerance=0.05):
    result = group.copy()
    result['nominal_coverage'] = result['quantile_name'].map({'q90': 0.90, 'q98': 0.98})
    result['coverage_floor'] = result['nominal_coverage'] - coverage_tolerance
    feasible = result[result['clean_coverage'] >= result['coverage_floor']].copy()
    fallback = feasible.empty
    if fallback:
        result['coverage_shortfall'] = np.maximum(
            result['coverage_floor'] - result['clean_coverage'],
            0.0,
        )
        shortfall = result['coverage_shortfall'].min()
        feasible = result[
            np.isclose(result['coverage_shortfall'], shortfall, rtol=0.0, atol=1e-12)
        ].copy()
    else:
        feasible['coverage_shortfall'] = 0.0
    feasible['own_sort'] = feasible['own_event_recall'].fillna(-1.0)
    feasible['cross_sort'] = feasible['cross_channel_cascade_events']
    feasible['cross_seconds_sort'] = feasible['cross_channel_suspicious_seconds']
    feasible['process_sort'] = feasible['process_false_alarm_events']
    feasible['process_seconds_sort'] = feasible['process_suspicious_seconds']
    selected = feasible.sort_values(
        [
            'own_sort',
            'cross_sort',
            'cross_seconds_sort',
            'process_sort',
            'process_seconds_sort',
            'clean_interval_score',
            'clean_mean_width',
            'model_type',
            'quantile_name',
            'calibration_method',
        ],
        ascending=[False, True, True, True, True, True, True, True, True, True],
        kind='mergesort',
    ).iloc[0]
    selected = selected.to_dict()
    selected['quality_feasible'] = bool(not fallback)
    selected['selection_fallback_used'] = bool(fallback)
    return selected


## 5. Валидационная проверка: собственные неисправности, ложные тревоги процесса и каскадные тревоги

Оценка использует события и маски. Собственная неисправность относится к текущему целевому каналу, проверка процесса считается только на чистых строках, каскадная тревога — на событиях неисправностей других каналов. Для тревоги сохраняется правило подтверждения 5 секунд.


In [8]:
PERSISTENCE_SECONDS = 5


def _runs(mask):
    indices = np.flatnonzero(np.asarray(mask, dtype=bool))
    if len(indices) == 0:
        return []
    starts = indices[np.r_[True, np.diff(indices) > 1]]
    ends = indices[np.r_[np.diff(indices) > 1, True]] + 1
    return [(int(start), int(end)) for start, end in zip(starts, ends)]


def _event_mask(frame, start_ts, end_ts):
    timestamps = pd.to_datetime(frame['timestamp'], utc=True)
    start = pd.Timestamp(start_ts)
    end = pd.Timestamp(end_ts)
    start = start.tz_localize('UTC') if start.tzinfo is None else start.tz_convert('UTC')
    end = end.tz_localize('UTC') if end.tzinfo is None else end.tz_convert('UTC')
    return ((timestamps >= start) & (timestamps < end)).to_numpy(dtype=bool)


def _after_timestamp(frame, mask, origin):
    if origin is None or pd.isna(origin):
        return np.asarray(mask, dtype=bool).copy()
    timestamp = pd.Timestamp(origin)
    timestamp = (
        timestamp.tz_localize('UTC')
        if timestamp.tzinfo is None
        else timestamp.tz_convert('UTC')
    )
    timestamps = pd.to_datetime(frame['timestamp'], utc=True)
    return np.asarray(mask, dtype=bool) & (timestamps >= timestamp).to_numpy(dtype=bool)


def _first_index(mask):
    indices = np.flatnonzero(np.asarray(mask, dtype=bool))
    return int(indices[0]) if len(indices) else None


def _delay_seconds(frame, index, origin):
    if index is None or origin is None or pd.isna(origin):
        return float('nan')
    timestamp = pd.Timestamp(origin)
    timestamp = (
        timestamp.tz_localize('UTC')
        if timestamp.tzinfo is None
        else timestamp.tz_convert('UTC')
    )
    timestamps = pd.to_datetime(frame['timestamp'], utc=True)
    return float((timestamps.iloc[index] - timestamp).total_seconds())


def _detect_branch(actual, intervals, feature_valid):
    actual = np.asarray(actual, dtype=np.float64)
    intervals = np.asarray(intervals, dtype=np.float64)
    feature_valid = np.asarray(feature_valid, dtype=bool)
    target_nan = ~np.isfinite(actual)
    input_nan = (~feature_valid) & ~target_nan
    bounds_invalid = ~np.isfinite(intervals).all(axis=1)
    outlier = np.zeros(len(actual), dtype=bool)
    suspicious = np.zeros(len(actual), dtype=bool)
    system_alarm = np.zeros(len(actual), dtype=bool)
    streak = 0
    for index in range(len(actual)):
        if target_nan[index]:
            system_alarm[index] = True
            streak = 0
            continue
        if input_nan[index] or bounds_invalid[index]:
            streak = 0
            continue
        is_outlier = bool(
            actual[index] < intervals[index, 0]
            or actual[index] > intervals[index, 2]
        )
        outlier[index] = is_outlier
        streak = streak + 1 if is_outlier else 0
        if streak >= PERSISTENCE_SECONDS:
            suspicious[index] = True
            system_alarm[index] = True
    return {
        'outlier_second': outlier,
        'suspicious': suspicious,
        'target_nan_guardrail': target_nan,
        'input_nan_guardrail': input_nan,
        'system_alarm': system_alarm,
    }


def _detect_consensus(actual, strict, full, strict_valid, full_valid):
    actual = np.asarray(actual, dtype=np.float64)
    strict = np.asarray(strict, dtype=np.float64)
    full = np.asarray(full, dtype=np.float64)
    strict_valid = np.asarray(strict_valid, dtype=bool)
    full_valid = np.asarray(full_valid, dtype=bool)
    target_nan = ~np.isfinite(actual)
    input_nan = (~strict_valid | ~full_valid) & ~target_nan
    bounds_valid = np.isfinite(strict).all(axis=1) & np.isfinite(full).all(axis=1)
    usable = ~target_nan & ~input_nan & bounds_valid
    strict_low = usable & (actual < strict[:, 0])
    strict_high = usable & (actual > strict[:, 2])
    full_low = usable & (actual < full[:, 0])
    full_high = usable & (actual > full[:, 2])
    outlier = (strict_low & full_low) | (strict_high & full_high)
    suspicious = np.zeros(len(actual), dtype=bool)
    system_alarm = np.zeros(len(actual), dtype=bool)
    streak = 0
    for index in range(len(actual)):
        if target_nan[index]:
            system_alarm[index] = True
            streak = 0
        elif input_nan[index] or not bounds_valid[index]:
            streak = 0
        elif outlier[index]:
            streak += 1
        else:
            streak = 0
        if streak >= PERSISTENCE_SECONDS:
            suspicious[index] = True
            system_alarm[index] = True
    return {
        'outlier_second': outlier,
        'suspicious': suspicious,
        'target_nan_guardrail': target_nan,
        'input_nan_guardrail': input_nan,
        'system_alarm': system_alarm,
    }


def evaluate_events(frame, masks, events, target, detector, selection_mask, feature_valid=None):
    labels = masks['label_state'].astype(str).to_numpy()
    fault_events = events[events['event_class'] == 'sensor_fault'].copy()
    process_events = events[events['event_class'].isin(['process', 'maintenance'])].copy()

    event_rows = []
    for record in fault_events.itertuples(index=False):
        interval = _event_mask(frame, record.start_ts, record.end_ts)
        after_confirmed = _after_timestamp(frame, interval, record.confirmed_ts)
        system = detector['system_alarm'] & after_confirmed
        suspicious = detector['suspicious'] & after_confirmed
        first_alarm = _first_index(system)
        first_suspicious = _first_index(suspicious)
        event_rows.append({
            'event_id': record.event_id,
            'target': target,
            'event_role': 'own' if record.sensor == target else 'cross',
            'fault_sensor': record.sensor,
            'fault_type': record.event_type,
            'detected_after_confirmed': first_alarm is not None,
            'detection_source': (
                'target_nan_guardrail'
                if first_alarm is not None and detector['target_nan_guardrail'][first_alarm]
                else 'suspicious' if first_alarm is not None else 'none'
            ),
            'sustained_alarm': first_suspicious is not None,
            'detection_delay_seconds': _delay_seconds(frame, first_alarm, record.confirmed_ts),
            'suspicious_detection_delay_seconds': _delay_seconds(
                frame, first_suspicious, record.confirmed_ts
            ),
            'alarm_rows': int(system.sum()),
            'suspicious_rows': int(suspicious.sum()),
            'target_nan_guardrail_rows': int(
                (detector['target_nan_guardrail'] & after_confirmed).sum()
            ),
        })

    process_rows = []
    no_sensor_fault = ~masks['sensor_fault_active'].astype(bool).to_numpy()
    for record in process_events.itertuples(index=False):
        interval = _event_mask(frame, record.start_ts, record.end_ts)
        healthy = interval & no_sensor_fault & (labels == 'CLEAN')
        suspicious = detector['suspicious'] & healthy
        process_rows.append({
            'event_id': record.event_id,
            'target': target,
            'event_type': record.event_type,
            'event_rows': int(healthy.sum()),
            'false_alarm': bool(suspicious.any()),
            'false_alarm_runs': int(len(_runs(suspicious))),
            'suspicious_alarm_seconds': int(suspicious.sum()),
        })

    cross_rows = []
    for record in fault_events.itertuples(index=False):
        if record.sensor == target:
            continue
        interval = _event_mask(frame, record.start_ts, record.end_ts)
        after_confirmed = _after_timestamp(frame, interval, record.confirmed_ts)
        suspicious = detector['suspicious'] & after_confirmed
        cross_rows.append({
            'event_id': record.event_id,
            'fault_sensor': record.sensor,
            'fault_type': record.event_type,
            'target': target,
            'cascade_false_alarm': bool(suspicious.any()),
            'cascade_false_alarm_runs': int(len(_runs(suspicious))),
            'cascade_suspicious_seconds': int(suspicious.sum()),
        })

    event_metrics = pd.DataFrame(event_rows)
    process_metrics = pd.DataFrame(process_rows)
    cross_metrics = pd.DataFrame(cross_rows)
    own = event_metrics[event_metrics['event_role'] == 'own']
    delays = own['detection_delay_seconds'].dropna().to_numpy(dtype=np.float64)
    return {
        'own_event_recall': float(own['detected_after_confirmed'].mean()) if len(own) else np.nan,
        'own_sustained_recall': float(own['sustained_alarm'].mean()) if len(own) else np.nan,
        'own_mean_detection_delay_seconds': float(np.mean(delays)) if len(delays) else np.nan,
        'process_false_alarm_events': int(process_metrics['false_alarm'].sum()) if len(process_metrics) else 0,
        'process_suspicious_seconds': int(process_metrics['suspicious_alarm_seconds'].sum()) if len(process_metrics) else 0,
        'cross_channel_cascade_events': int(cross_metrics['cascade_false_alarm'].sum()) if len(cross_metrics) else 0,
        'cross_channel_suspicious_seconds': int(cross_metrics['cascade_suspicious_seconds'].sum()) if len(cross_metrics) else 0,
        'feature_valid_rate': float(np.mean(feature_valid)) if feature_valid is not None else np.nan,
        'clean_outlier_seconds': int((detector['outlier_second'] & selection_mask).sum()),
        'event_metrics': event_metrics,
        'process_metrics': process_metrics,
        'cross_channel_metrics': cross_metrics,
    }


## 6. Обучение кандидатов и выбор по валидационной выборке

Для каждого целевого канала и ветки проверяются линейные модели q90/q98 и CatBoost q98 с глобальной калибровкой и калибровкой по режимам. Сначала применяется порог покрытия, затем кандидаты сравниваются по обнаружению собственных неисправностей, каскадным тревогам, ложным тревогам процесса, оценке интервала и его ширине.


In [9]:
def _candidate_key(row):
    return (
        f"{row['target']}|{row['branch']}|{row['model_type']}|"
        f"{row['quantile_name']}|{row['calibration_method']}"
    )


if training_enabled:
    modes = validation['X01'].to_numpy(dtype=np.int64)
    calibration_rows = np.arange(len(validation)) < int(
        len(validation) * CALIBRATION_FRACTION
    )
    selection_rows = ~calibration_rows
    candidate_cache = {}
    candidate_rows = []

    for target in targets:
        for branch in branches:
            features = feature_policies[target][branch]
            specifications = [
                ('linear_quantile', 'q90'),
                ('linear_quantile', 'q98'),
                ('catboost_multi_quantile', 'q98'),
            ]
            for model_type, quantile_name in specifications:
                levels = QUANTILE_SETS[quantile_name]
                fitted = fit_candidate(train_fit, target, features, model_type, levels)
                raw, valid = predict_candidate(fitted, validation, features)
                base_key = f'{target}|{branch}|{model_type}|{quantile_name}'
                candidate_cache[base_key] = {
                    'fitted': fitted,
                    'features': features,
                    'raw': raw,
                    'valid': valid,
                }
                actual = validation[target].to_numpy(dtype=np.float64)
                alpha = 1.0 - levels[2] + levels[0]

                for method in CALIBRATION_METHODS:
                    valid_calibration = calibration_rows & valid
                    calibration = fit_calibration(
                        actual[valid_calibration],
                        raw[valid_calibration],
                        modes[valid_calibration],
                        method,
                        alpha,
                    )
                    calibrated = apply_calibration(raw, modes, calibration)
                    interval = interval_metrics(
                        actual[selection_rows],
                        calibrated[selection_rows],
                        alpha,
                    )
                    detector = _detect_branch(actual, calibrated, valid)
                    quality = evaluate_events(
                        validation,
                        validation_masks,
                        validation_events,
                        target,
                        detector,
                        selection_rows,
                        valid,
                    )
                    key = f'{base_key}|{method}'
                    candidate_cache[key] = {
                        **candidate_cache[base_key],
                        'calibration': calibration,
                        'calibrated': calibrated,
                    }
                    candidate_rows.append({
                        'candidate_id': key,
                        'target': target,
                        'branch': branch,
                        'model_type': model_type,
                        'quantile_name': quantile_name,
                        'calibration_method': method,
                        'alpha': alpha,
                        'clean_coverage': interval['coverage'],
                        'clean_outside_rate': interval['outside_rate'],
                        'clean_mean_width': interval['mean_interval_width'],
                        'clean_interval_score': interval['mean_interval_score'],
                        **quality,
                    })

    candidate_metrics = pd.DataFrame(candidate_rows)
    assert len(candidate_metrics) == 72
    branch_selection = pd.DataFrame([
        {
            **select_candidate(group),
            'branch': branch,
        }
        for (target, branch), group in candidate_metrics.groupby(
            ['target', 'branch'],
            sort=True,
        )
    ])
    display(candidate_metrics[[
        'target',
        'branch',
        'model_type',
        'quantile_name',
        'calibration_method',
        'clean_coverage',
        'clean_mean_width',
        'clean_interval_score',
        'own_event_recall',
        'process_false_alarm_events',
        'cross_channel_cascade_events',
    ]].head(16).round(4))
    display(branch_selection[[
        'target',
        'branch',
        'model_type',
        'quantile_name',
        'calibration_method',
        'clean_coverage',
        'clean_mean_width',
        'clean_interval_score',
    ]])
else:
    candidate_cache = {}
    candidate_metrics = pd.DataFrame()
    branch_selection = pd.DataFrame()


## 7. Валидационная проверка: сравнение веток и Consensus

Перебор кандидатов сравнивается по событиям и метрикам. При отключённом обучении используются ранее рассчитанные результаты, чтобы не запускать длительное обучение повторно.


In [10]:
SOURCE_TARGETS = None
if METRICS_ROOT is not None:
    observed_path = SPLIT_DIRS['train'] / 'observed.parquet'
    if observed_path.is_file():
        import pyarrow.parquet as pq

        source_columns = [
            column
            for column in pq.ParquetFile(observed_path).schema.names
            if column != 'timestamp'
        ]
        SOURCE_TARGETS = source_columns[5:11]


def _map_target(values):
    if SOURCE_TARGETS is None:
        return values.map(lambda value: value if value in targets else np.nan)
    return values.map(dict(zip(SOURCE_TARGETS, targets)))


def _sort_targets(frame):
    if frame.empty:
        return frame
    order = {target: index for index, target in enumerate(targets)}
    result = frame.copy()
    result['_target_order'] = result['target'].map(order)
    return result.sort_values('_target_order', kind='mergesort').drop(columns='_target_order').reset_index(drop=True)


def _build_system_tables(frame, masks, events, selection_mask):
    selection_rows = []
    branch_rows = []
    own_rows = []
    for target in targets:
        strict_row = branch_selection[
            (branch_selection['target'] == target)
            & (branch_selection['branch'] == 'strict_robust')
        ].iloc[0]
        full_row = branch_selection[
            (branch_selection['target'] == target)
            & (branch_selection['branch'] == 'full_telemetry')
        ].iloc[0]
        strict_result = candidate_cache[_candidate_key(strict_row)]
        full_result = candidate_cache[_candidate_key(full_row)]
        actual = frame[target].to_numpy(dtype=np.float64)
        detector = _detect_consensus(
            actual,
            strict_result['calibrated'],
            full_result['calibrated'],
            strict_result['valid'],
            full_result['valid'],
        )
        interval = np.column_stack((
            np.minimum(strict_result['calibrated'][:, 0], full_result['calibrated'][:, 0]),
            (strict_result['calibrated'][:, 1] + full_result['calibrated'][:, 1]) / 2.0,
            np.maximum(strict_result['calibrated'][:, 2], full_result['calibrated'][:, 2]),
        ))
        quality = evaluate_events(
            frame,
            masks,
            events,
            target,
            detector,
            selection_mask,
            strict_result['valid'] & full_result['valid'],
        )
        interval_quality = interval_metrics(
            actual[selection_mask],
            interval[selection_mask],
            0.02,
        )
        selection_rows.append({
            'target': target,
            'architecture': 'robust_consensus',
            'quantile_name': strict_row['quantile_name'],
            'calibration': (
                f"strict={strict_row['calibration_method']}; "
                f"full={full_row['calibration_method']}"
            ),
            'clean_coverage': interval_quality['coverage'],
            'clean_mean_width': interval_quality['mean_interval_width'],
            'clean_interval_score': interval_quality['mean_interval_score'],
            'own_event_recall': quality['own_event_recall'],
            'process_false_alarm_events': quality['process_false_alarm_events'],
            'process_suspicious_seconds': quality['process_suspicious_seconds'],
            'cascade_events': quality['cross_channel_cascade_events'],
            'cascade_seconds': quality['cross_channel_suspicious_seconds'],
        })
        for row, variant in ((full_row, 'Full'), (strict_row, 'Strict')):
            branch_rows.append({
                'target': target,
                'variant': variant,
                'process_false_alarm_events': int(row['process_false_alarm_events']),
                'process_suspicious_seconds': int(row['process_suspicious_seconds']),
                'cascade_false_alarm_events': int(row['cross_channel_cascade_events']),
                'cascade_suspicious_seconds': int(row['cross_channel_suspicious_seconds']),
            })
        own = quality['event_metrics']
        own = own[own['event_role'] == 'own']
        delays = own['detection_delay_seconds'].dropna()
        sources = own['detection_source'].replace({
            'target_nan_guardrail': 'защитное правило NaN',
            'suspicious': 'срабатывание',
            'none': 'нет срабатывания',
        }).drop_duplicates().tolist()
        own_rows.append({
            'target': target,
            'own_fault_events': len(own),
            'detected_events': int(own['detected_after_confirmed'].sum()),
            'own_event_recall': float(own['detected_after_confirmed'].mean()),
            'sustained_events': int(own['sustained_alarm'].sum()),
            'mean_detection_delay_seconds': float(delays.mean()) if len(delays) else np.nan,
            'detection_source': ', '.join(sources),
        })
        branch_rows.append({
            'target': target,
            'variant': 'Consensus',
            'process_false_alarm_events': quality['process_false_alarm_events'],
            'process_suspicious_seconds': quality['process_suspicious_seconds'],
            'cascade_false_alarm_events': quality['cross_channel_cascade_events'],
            'cascade_suspicious_seconds': quality['cross_channel_suspicious_seconds'],
        })
    return (
        pd.DataFrame(selection_rows),
        pd.DataFrame(branch_rows),
        pd.DataFrame(own_rows),
    )

validation_selection = pd.DataFrame()
validation_branch_comparison = pd.DataFrame()
validation_own = pd.DataFrame()
selected_model_rows = pd.DataFrame()
final_compare_raw = None
final_event_raw = None

if training_enabled:
    validation_selection, validation_branch_comparison, validation_own = _build_system_tables(
        validation,
        validation_masks,
        validation_events,
        selection_rows,
    )
    selected_model_rows = branch_selection.copy()
elif METRICS_ROOT is not None:
    selection_raw = pd.read_csv(METRICS_ROOT / 'model_v2_s2_selection.csv')
    branch_raw = pd.read_csv(METRICS_ROOT / 'model_v2_s2_branch_selection.csv')
    event_raw = pd.read_csv(METRICS_ROOT / 'model_v2_s2_event_metrics.csv')
    final_compare_raw = pd.read_csv(METRICS_ROOT / 'model_v2_s3_validation_test_comparison.csv')
    final_event_raw = pd.read_csv(METRICS_ROOT / 'model_v2_s3_event_metrics.csv')

    selection_raw['target'] = _map_target(selection_raw['target'])
    validation_selection = selection_raw.rename(columns={
        'selected_architecture': 'architecture',
        'selected_quantile_name': 'quantile_name',
        'selected_calibration_method': 'calibration',
        'clean_coverage': 'clean_coverage',
        'clean_mean_width': 'clean_mean_width',
        'clean_interval_score': 'clean_interval_score',
        'own_event_recall': 'own_event_recall',
        'process_false_alarm_events': 'process_false_alarm_events',
        'process_suspicious_seconds': 'process_suspicious_seconds',
        'cross_channel_cascade_events': 'cascade_events',
        'cross_channel_suspicious_seconds': 'cascade_seconds',
    })[[
        'target',
        'architecture',
        'quantile_name',
        'calibration',
        'clean_coverage',
        'clean_mean_width',
        'clean_interval_score',
        'own_event_recall',
        'process_false_alarm_events',
        'process_suspicious_seconds',
        'cascade_events',
        'cascade_seconds',
    ]]

    branch_raw['target'] = _map_target(branch_raw['target'])
    branch_rows = []
    for row in branch_raw.itertuples(index=False):
        branch_rows.append({
            'target': row.target,
            'variant': 'Full' if row.policy == 'full_telemetry' else 'Strict',
            'process_false_alarm_events': int(row.selected_process_false_alarm_events),
            'process_suspicious_seconds': int(row.selected_process_suspicious_seconds),
            'cascade_false_alarm_events': int(row.selected_cross_channel_cascade_events),
            'cascade_suspicious_seconds': int(row.selected_cross_channel_suspicious_seconds),
        })
    branch_rows.extend(
        validation_selection.assign(variant='Consensus').rename(columns={
            'cascade_events': 'cascade_false_alarm_events',
            'cascade_seconds': 'cascade_suspicious_seconds',
        })[[
            'target',
            'variant',
            'process_false_alarm_events',
            'process_suspicious_seconds',
            'cascade_false_alarm_events',
            'cascade_suspicious_seconds',
        ]].to_dict('records')
    )
    validation_branch_comparison = pd.DataFrame(branch_rows)

    event_raw['target'] = _map_target(event_raw['target'])
    own_raw = event_raw[
        (event_raw['architecture'] == 'robust_consensus')
        & (event_raw['policy'] == 'strict_robust_mandatory')
        & (event_raw['event_role'] == 'own')
    ]
    own_rows = []
    for target, group in own_raw.groupby('target', sort=False):
        sources = group['detection_source'].replace({
            'target_nan_guardrail': 'защитное правило NaN',
            'suspicious': 'срабатывание',
            'none': 'нет срабатывания',
        }).drop_duplicates().tolist()
        delays = group['detection_delay_seconds'].dropna()
        own_rows.append({
            'target': target,
            'own_fault_events': len(group),
            'detected_events': int(group['detected_after_confirmed'].sum()),
            'own_event_recall': float(group['detected_after_confirmed'].mean()),
            'sustained_events': int(group['sustained_alarm'].sum()),
            'mean_detection_delay_seconds': float(delays.mean()) if len(delays) else np.nan,
            'detection_source': ', '.join(sources),
        })
    validation_own = pd.DataFrame(own_rows)
    selected_model_rows = branch_raw.copy()

validation_selection = _sort_targets(validation_selection)
validation_branch_comparison = _sort_targets(validation_branch_comparison)
validation_own = _sort_targets(validation_own)
if not validation_selection.empty:
    display(validation_selection.round(6))


,target,architecture,quantile_name,calibration,clean_coverage,clean_mean_width,clean_interval_score,own_event_recall,process_false_alarm_events,process_suspicious_seconds,cascade_events,cascade_seconds
0,X06,robust_consensus,q98,strict=global_symmetric_conformal;full=operati...,0.993688,0.473209,0.485697,1.0,4,126,0,0
1,X07,robust_consensus,q98,strict=global_symmetric_conformal;full=operati...,0.994499,0.556690,0.570523,1.0,3,69,0,0
2,X08,robust_consensus,q98,strict=global_symmetric_conformal;full=operati...,0.999281,0.784507,0.786194,1.0,5,267,0,0
3,X09,robust_consensus,q98,strict=operating_mode_mondrian;full=operating_...,1.000000,0.343763,0.343763,1.0,2,119,0,0
4,X10,robust_consensus,q98,strict=global_symmetric_conformal;full=operati...,0.996210,0.521821,0.530019,1.0,4,245,0,0
5,X11,robust_consensus,q98,strict=operating_mode_mondrian;full=operating_...,0.987247,4.418325,4.630007,1.0,3,92,0,0


### Full / Strict / Consensus

Для каждого X06–X11 показаны ложные тревоги процесса и каскадные ложные тревоги. Consensus требует согласия Strict и Full по одной стороне границы.


In [11]:
if not validation_branch_comparison.empty:
    display(validation_branch_comparison)


,target,variant,process_false_alarm_events,process_suspicious_seconds,cascade_false_alarm_events,cascade_suspicious_seconds
0,X06,Full,7,1266,3,22307
1,X06,Strict,7,1903,0,0
2,X06,Consensus,4,126,0,0
3,X07,Full,7,808,1,6087
4,X07,Strict,7,2008,1,2
5,X07,Consensus,3,69,0,0
6,X08,Full,7,2664,5,29013
7,X08,Strict,6,949,0,0
8,X08,Consensus,5,267,0,0
9,X09,Full,6,4979,3,26


### Обнаружение собственных неисправностей на валидационной выборке

Метрика `own_event_recall` считается на уровне эпизодов неисправности: она отвечает только на вопрос, был ли эпизод обнаружен хотя бы один раз. Это не доля обнаруженных секунд неисправности.


In [12]:
if not validation_own.empty:
    display(validation_own)
    assert int(validation_own['own_fault_events'].sum()) == 7
    assert int(validation_own['detected_events'].sum()) == 7


,target,own_fault_events,detected_events,own_event_recall,sustained_events,mean_detection_delay_seconds,detection_source
0,X06,1,1,1.0,1,354.0,срабатывание
1,X07,2,2,1.0,2,0.0,срабатывание
2,X08,1,1,1.0,0,0.0,защитное правило NaN
3,X09,1,1,1.0,1,0.0,срабатывание
4,X10,1,1,1.0,1,91.0,срабатывание
5,X11,1,1,1.0,1,4391.0,срабатывание


## 8. Итоговая проверка

Итоговая тестовая выборка открывается после выбора конфигураций. Она не участвует в калибровке и выборе модели.


In [13]:
final_test_results = pd.DataFrame()
final_event_metrics = pd.DataFrame()

if training_enabled:
    final_test, final_test_masks, final_test_events = _load_split('final_test')
    final_clean = final_test['label_state'].eq('CLEAN').to_numpy(dtype=bool)
    final_rows = []
    event_frames = []
    for target in targets:
        strict_row = branch_selection[
            (branch_selection['target'] == target)
            & (branch_selection['branch'] == 'strict_robust')
        ].iloc[0]
        full_row = branch_selection[
            (branch_selection['target'] == target)
            & (branch_selection['branch'] == 'full_telemetry')
        ].iloc[0]
        strict_result = candidate_cache[_candidate_key(strict_row)]
        full_result = candidate_cache[_candidate_key(full_row)]
        strict_raw, strict_valid = predict_candidate(
            strict_result['fitted'],
            final_test,
            strict_result['features'],
        )
        full_raw, full_valid = predict_candidate(
            full_result['fitted'],
            final_test,
            full_result['features'],
        )
        modes = final_test['X01'].to_numpy(dtype=np.int64)
        strict_calibrated = apply_calibration(strict_raw, modes, strict_result['calibration'])
        full_calibrated = apply_calibration(full_raw, modes, full_result['calibration'])
        actual = final_test[target].to_numpy(dtype=np.float64)
        detector = _detect_consensus(
            actual,
            strict_calibrated,
            full_calibrated,
            strict_valid,
            full_valid,
        )
        interval = np.column_stack((
            np.minimum(strict_calibrated[:, 0], full_calibrated[:, 0]),
            (strict_calibrated[:, 1] + full_calibrated[:, 1]) / 2.0,
            np.maximum(strict_calibrated[:, 2], full_calibrated[:, 2]),
        ))
        quality = evaluate_events(
            final_test,
            final_test_masks,
            final_test_events,
            target,
            detector,
            final_clean,
            strict_valid & full_valid,
        )
        metrics = interval_metrics(actual[final_clean], interval[final_clean], 0.02)
        validation_row = validation_selection[
            validation_selection['target'] == target
        ].iloc[0]
        final_rows.append({
            'target': target,
            'validation_coverage': validation_row['clean_coverage'],
            'final_test_coverage': metrics['coverage'],
            'final_test_width': metrics['mean_interval_width'],
            'final_test_interval_score': metrics['mean_interval_score'],
            'final_process_false_alarm_events': quality['process_false_alarm_events'],
            'final_process_suspicious_seconds': quality['process_suspicious_seconds'],
            'final_cascade_events': quality['cross_channel_cascade_events'],
            'final_cascade_seconds': quality['cross_channel_suspicious_seconds'],
            'final_own_event_recall': quality['own_event_recall'],
        })
        event_frames.append(quality['event_metrics'])
    final_test_results = pd.DataFrame(final_rows)
    final_event_metrics = pd.concat(event_frames, ignore_index=True)
elif final_compare_raw is not None:
    final_test_results = final_compare_raw.copy()
    final_test_results['target'] = _map_target(final_test_results['target'])
    final_test_results = final_test_results.rename(columns={
        'development_validation_clean_coverage': 'validation_coverage',
        'final_test_clean_coverage': 'final_test_coverage',
        'final_test_mean_interval_width': 'final_test_width',
        'final_test_interval_score': 'final_test_interval_score',
        'final_process_false_alarm_events': 'final_process_false_alarm_events',
        'final_process_suspicious_seconds': 'final_process_suspicious_seconds',
        'final_cross_cascade_events': 'final_cascade_events',
        'final_cross_suspicious_seconds': 'final_cascade_seconds',
        'final_own_event_recall': 'final_own_event_recall',
    })[[
        'target',
        'validation_coverage',
        'final_test_coverage',
        'final_test_width',
        'final_test_interval_score',
        'final_process_false_alarm_events',
        'final_process_suspicious_seconds',
        'final_cascade_events',
        'final_cascade_seconds',
        'final_own_event_recall',
    ]]

final_test_results = _sort_targets(final_test_results)
if not final_test_results.empty:
    display(final_test_results.round(6))


,target,validation_coverage,final_test_coverage,final_test_width,final_test_interval_score,final_process_false_alarm_events,final_process_suspicious_seconds,final_cascade_events,final_cascade_seconds,final_own_event_recall
0,X06,0.993688,0.983947,0.474919,0.517700,3,105,1,3,1.0
1,X07,0.994499,0.984139,0.558568,0.609354,1,63,1,32,1.0
2,X08,0.999281,0.992340,0.785908,0.812269,2,33,3,49,1.0
3,X09,1.000000,0.997727,0.343951,0.346359,0,0,0,0,1.0
4,X10,0.996210,0.987800,0.523594,0.560062,5,210,0,0,1.0
5,X11,0.987247,0.979158,4.430529,4.925962,2,247,0,0,0.0


### Результат обнаружения собственных неисправностей на итоговой тестовой выборке

В этом наборе семь эпизодов собственных неисправностей: X07 содержит два. X08 обнаруживается сразу по защитному правилу NaN; 900 с — длительность защитного состояния внутри события неисправности, а не задержка обнаружения.


In [14]:
final_own = pd.DataFrame()
if training_enabled:
    own_raw = final_event_metrics[final_event_metrics['event_role'] == 'own']
else:
    own_raw = pd.DataFrame()
    if final_event_raw is not None:
        own_raw = final_event_raw.copy()
        own_raw['target'] = _map_target(own_raw['target'])
        own_raw = own_raw[own_raw['event_role'] == 'own']

if not own_raw.empty:
    own_rows = []
    for target, group in own_raw.groupby('target', sort=False):
        delays = group['detection_delay_seconds'].dropna()
        sources = group['detection_source'].replace({
            'target_nan_guardrail': 'защитное правило NaN',
            'suspicious': 'срабатывание',
            'none': 'нет срабатывания',
        }).drop_duplicates().tolist()
        own_rows.append({
            'target': target,
            'own_fault_events': len(group),
            'detected_events': int(group['detected_after_confirmed'].sum()),
            'own_event_recall': float(group['detected_after_confirmed'].mean()),
            'sustained_events': int(group['sustained_alarm'].sum()),
            'mean_detection_delay_seconds': float(delays.mean()) if len(delays) else np.nan,
            'target_nan_guardrail_seconds': int(group['target_nan_guardrail_seconds'].sum()),
            'result': ', '.join(sources),
        })
    final_own = pd.DataFrame(own_rows)

final_own = _sort_targets(final_own)
if not final_own.empty:
    display(final_own)
    assert int(final_own['own_fault_events'].sum()) == 7
    assert int(final_own['detected_events'].sum()) == 6
    assert int(final_own['sustained_events'].sum()) == 5
    assert int(final_own.loc[final_own['target'] == 'X11', 'detected_events'].iloc[0]) == 0
    assert int(final_own.loc[final_own['target'] == 'X08', 'target_nan_guardrail_seconds'].iloc[0]) == 900


,target,own_fault_events,detected_events,own_event_recall,sustained_events,mean_detection_delay_seconds,target_nan_guardrail_seconds,result
0,X06,1,1,1.0,1,4113.0,0,срабатывание
1,X07,2,2,1.0,2,26.0,0,срабатывание
2,X08,1,1,1.0,0,0.0,900,защитное правило NaN
3,X09,1,1,1.0,1,0.0,0,срабатывание
4,X10,1,1,1.0,1,35.0,0,срабатывание
5,X11,1,0,0.0,0,NaN,0,нет срабатывания


## 9. График интервалов на валидационной выборке

График показывает фактическое значение и нижнюю / центральную / верхнюю границы выбранного интервала Consensus на второй половине валидационной выборки.


In [15]:
if training_enabled and not candidate_metrics.empty:
    plot_target = targets[0]
    plot_row = candidate_metrics[
        (candidate_metrics['target'] == plot_target)
        & (candidate_metrics['branch'] == 'full_telemetry')
    ].sort_values(
        ['clean_interval_score', 'clean_mean_width']
    ).iloc[0]
    plot_key = plot_row['candidate_id']
    plot_interval = candidate_cache[plot_key]['calibrated']
    plot_actual = validation[plot_target].to_numpy(dtype=float)
    plot_axis = np.arange(len(validation))

    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(
        plot_axis[selection_rows],
        plot_actual[selection_rows],
        color='#1d5f9e',
        linewidth=1,
        label=f'фактическое значение {plot_target}',
    )
    ax.fill_between(
        plot_axis[selection_rows],
        plot_interval[selection_rows, 0],
        plot_interval[selection_rows, 2],
        color='#23824b',
        alpha=0.18,
        label=f"{plot_row['model_type']} {plot_row['quantile_name']}",
    )
    ax.set_xlabel('секунда валидационной выборки')
    ax.set_ylabel(plot_target)
    ax.legend()
    fig.tight_layout()
    plt.show()


## 10. Выбор решения

Выбор модели формируется по метрикам кандидатов. Порог покрытия отбрасывает слишком узкие варианты, затем применяются приоритеты обнаружения собственных неисправностей, каскадных тревог, ложных тревог процесса, оценки интервала и его ширины.


In [16]:
if not selected_model_rows.empty:
    if training_enabled:
        selected_view = selected_model_rows[[
            'target',
            'branch',
            'model_type',
            'quantile_name',
            'calibration_method',
            'clean_coverage',
            'clean_mean_width',
            'clean_interval_score',
        ]]
    else:
        selected_view = selected_model_rows[[
            'target',
            'policy',
            'selected_model_type',
            'selected_quantile_name',
            'selected_calibration_method',
            'selected_clean_coverage',
            'selected_clean_mean_width',
            'selected_clean_interval_score',
        ]]
    display(selected_view)


,target,policy,selected_model_type,selected_quantile_name,selected_calibration_method,selected_clean_coverage,selected_clean_mean_width,selected_clean_interval_score
0,X06,full_telemetry,catboost_multi_quantile,q98,operating_mode_mondrian,0.986679,0.318213,0.349507
1,X06,strict_robust,linear_quantile,q98,global_symmetric_conformal,0.993688,0.473209,0.485697
2,X09,full_telemetry,catboost_multi_quantile,q98,operating_mode_mondrian,0.985419,0.240742,0.267151
3,X09,strict_robust,linear_quantile,q98,operating_mode_mondrian,1.000000,0.343763,0.343763
4,X11,full_telemetry,catboost_multi_quantile,q98,operating_mode_mondrian,0.666486,2.786200,17.011633
5,X11,strict_robust,linear_quantile,q98,operating_mode_mondrian,0.987247,4.418325,4.630007
6,X08,full_telemetry,catboost_multi_quantile,q98,operating_mode_mondrian,0.987121,0.357358,0.401479
7,X08,strict_robust,linear_quantile,q98,global_symmetric_conformal,0.999281,0.784507,0.786194
8,X07,full_telemetry,catboost_multi_quantile,q98,operating_mode_mondrian,0.988286,0.383140,0.411602
9,X07,strict_robust,linear_quantile,q98,global_symmetric_conformal,0.994499,0.556690,0.570523


## 11. Сохранение выбранных моделей

Выбранные ветки сохраняются в JSON вместе с описанием для запуска. 

In [17]:
def _jsonable(value):
    if isinstance(value, dict):
        return {str(key): _jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


def _save_linear_model(fitted, target, branch, quantile_name, levels, path):
    state = fitted['state']
    payload = {
        'target': target,
        'policy': branch,
        'model_type': 'linear_quantile',
        'quantile_name': quantile_name,
        'quantile_levels': list(levels),
        'encoder': {
            'features': feature_policies[target][branch],
            'numeric_features': state['numeric'],
            'numeric_means': state['means'],
            'numeric_stds': state['stds'],
            'mode_categories': state['categories'],
            'design_features': [
                'bias',
                *state['numeric'],
                *[f"X01=={category}" for category in state['categories'][1:]],
            ],
        },
        'coefficients': fitted['coefficients'],
        **LINEAR_PARAMS,
    }
    path.write_text(
        json.dumps(_jsonable(payload), ensure_ascii=False, indent=2, sort_keys=True) + chr(10),
        encoding='utf-8',
    )


def save_selected_models(selection, cache, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest = {
        'targets': list(targets),
        'branches': list(branches),
        'models': {target: {} for target in targets},
    }
    branch_names = {'strict_robust': 'strict', 'full_telemetry': 'full'}
    for row in selection.to_dict('records'):
        target = row['target']
        branch = row['branch']
        levels = QUANTILE_SETS[row['quantile_name']]
        result = cache[_candidate_key(row)]
        filename = f"{target}_{branch_names[branch]}.json"
        path = output_dir / filename
        if result['fitted']['kind'] == 'linear':
            _save_linear_model(
                result['fitted'],
                target,
                branch,
                row['quantile_name'],
                levels,
                path,
            )
        else:
            result['fitted']['model'].save_model(str(path), format='json')
        manifest['models'][target][branch] = {
            'model_path': filename,
            'model_type': (
                'linear_quantile'
                if result['fitted']['kind'] == 'linear'
                else 'catboost_multi_quantile'
            ),
            'features': feature_policies[target][branch],
            'quantile_levels': list(levels),
            'quantile_name': row['quantile_name'],
            'calibration': result['calibration'],
        }
    (output_dir / 'model_info.json').write_text(
        json.dumps(_jsonable(manifest), ensure_ascii=False, indent=2) + chr(10),
        encoding='utf-8',
    )
    return manifest


if EXPORT_MODELS and training_enabled:
    saved_manifest = save_selected_models(branch_selection, candidate_cache, ROOT / 'models')
